# TCC — Colab: mesmo pipeline do ambiente local

Os parâmetros não ficam mais duplicados no notebook. A fonte única de configuração é `configs/runtime.yaml`. O Colab usa o perfil `colab`; o Streamlit/terminal local usa o perfil `local`. Ambos chamam `scripts/train_experiments.py`.


## 1. Bootstrap: GPU, Drive e repositório


In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
from google.colab import drive, userdata

subprocess.run(['nvidia-smi'], check=False)
drive.mount('/content/drive')

REPO = Path('/content/TCC_Code_Test')
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('GITHUB_TOKEN não encontrado em Colab > Secrets.')
netrc = Path.home() / '.netrc'
netrc.write_text(f'machine github.com\nlogin x-access-token\npassword {token}\n', encoding='utf-8')
netrc.chmod(0o600)
try:
    if REPO.exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', 'https://github.com/ferreis/TCC_Code_Test.git', str(REPO)], check=True)
finally:
    netrc.unlink(missing_ok=True)
os.chdir(REPO)
print('Commit:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())


## 2. Instalar dependências do projeto


In [ ]:
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', 'requirements-colab.txt'], check=True)


## 3. Carregar a configuração compartilhada e validar YOLO-Master


In [ ]:
from tcc_discovery.runtime_config import load_runtime_config, runtime_profile

RUNTIME_CONFIG = 'configs/runtime.yaml'
PROFILE = 'colab'
runtime = load_runtime_config(RUNTIME_CONFIG)
_, profile_cfg = runtime_profile(runtime, PROFILE)
print('Perfil:', PROFILE)
print('Treinamento:', {k: profile_cfg.get(k) for k in ['batch','workers','cache','imgsz','epochs','device']})

yolo_master_dir = str(profile_cfg.get('yolo_master_dir', '/content/YOLO-Master'))
subprocess.run([sys.executable, 'scripts/setup_yolo_master.py', '--dir', yolo_master_dir], check=True)
check = 'import importlib.util, ultralytics; print(ultralytics.__version__); assert importlib.util.find_spec("ultralytics.utils.lora")'
subprocess.run([sys.executable, '-c', check], check=True)

import torch, importlib.util, ultralytics
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU:', p.name, '| VRAM:', round(p.total_memory / 1024**3, 2), 'GB')
print('YOLO-Master:', ultralytics.__version__, '| LoRA:', importlib.util.find_spec('ultralytics.utils.lora') is not None)


## 4. Persistir dados/resultados no Drive


In [ ]:
persist_root = Path(str(profile_cfg.get('persist_root', '/content/drive/MyDrive/TCC_Code_Test_runtime')))
for name in ['data', 'workspace', 'exports', 'weights']:
    target = persist_root / name
    target.mkdir(parents=True, exist_ok=True)
    link = REPO / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        if link.is_dir():
            shutil.copytree(link, target, dirs_exist_ok=True)
            shutil.rmtree(link)
        else:
            link.unlink()
    link.symlink_to(target, target_is_directory=True)
print('Persistência:', persist_root)


## 5. Baixar datasets configurados no Roboflow


In [ ]:
rf_cfg = runtime.get('roboflow', {})
secret_name = (runtime.get('secrets', {}) or {}).get('roboflow', 'ROBOFLOW_API_KEY')
rf_key = userdata.get(secret_name)
if not rf_key:
    raise RuntimeError(f'{secret_name} não encontrado em Colab > Secrets.')
os.environ['ROBOFLOW_API_KEY'] = rf_key

def download_rf(spec):
    if not spec.get('download', False):
        return
    project = str(spec['project'])
    if project.startswith('COLOQUE_'):
        raise ValueError(f'Preencha o slug Roboflow em configs/runtime.yaml: {project}')
    cmd = [sys.executable, 'scripts/download_roboflow.py',
           '--workspace', str(rf_cfg['workspace']), '--project', project,
           '--version', str(spec.get('version', 1)), '--format', str(rf_cfg.get('format', 'yolo26')),
           '--output', str(spec['output'])]
    if spec.get('overwrite', False): cmd.append('--overwrite')
    subprocess.run(cmd, check=True)

download_rf(rf_cfg.get('general', {}))
for stage_id, spec in (rf_cfg.get('specialists', {}) or {}).items():
    if spec.get('download', False):
        print('Dataset especialista:', stage_id)
        download_rf(spec)


## 6. SAM e clustering — opcionais conforme configs/runtime.yaml


In [ ]:
sam_cfg = runtime.get('sam', {}) or {}
if sam_cfg.get('download', False):
    subprocess.run([sys.executable, 'scripts/download_sam.py'], check=True)

cluster_cfg = runtime.get('clustering', {}) or {}
if cluster_cfg.get('enabled', False):
    from tcc_discovery.box_clustering import BoxClusteringConfig, run_box_clustering
    cfg = BoxClusteringConfig(
        context_scale=float(cluster_cfg.get('context_scale', 1.8)),
        min_crop_size_px=int(cluster_cfg.get('min_crop_size_px', 64)),
        min_cluster_size=int(cluster_cfg.get('min_cluster_size', 6)),
        embedding_batch_size=int(cluster_cfg.get('embedding_batch_size', 8)),
        dino_cpu_fallback_on_oom=bool(cluster_cfg.get('cpu_fallback_on_oom', True)),
    )
    output = Path(str(cluster_cfg.get('output', 'workspace/bbox_clusters_colab')))
    if not (output / 'catalog.json').exists():
        run_box_clustering(str(cluster_cfg['source']), output, cfg, overwrite=False)


## 7. Helper único de treinamento
O helper apenas chama a mesma CLI usada no computador local e no frontend Streamlit.


In [ ]:
def run_stage(stage_id, resume=False, dry_run=False):
    cmd = [sys.executable, 'scripts/train_experiments.py',
           '--runtime-config', RUNTIME_CONFIG, '--profile', PROFILE, '--only', stage_id]
    if resume: cmd.append('--resume')
    if dry_run: cmd.append('--dry-run')
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)

subprocess.run([sys.executable, 'scripts/train_experiments.py', '--runtime-config', RUNTIME_CONFIG, '--profile', PROFILE, '--group', 'all', '--list'], check=True)


## 8. Smoke Test — detecção


In [ ]:
run_stage('smoke_det_yolo26n')


## 9. Modelo Geral — detecção


In [ ]:
run_stage('det_general')


## 10. LoRA especialista — escolha uma etapa


In [ ]:
LORA_STAGE = 'det_lora_01'  # det_lora_01 ... det_lora_05
run_stage(LORA_STAGE)


## 11. Full Fine-Tune de controle da mesma LoRA


In [ ]:
FULLFT_STAGE = LORA_STAGE.replace('det_lora_', 'det_fullft_')
run_stage(FULLFT_STAGE)


## 12. Segmentação — quando o dataset estiver pronto


In [ ]:
seg_yaml = Path(str((runtime.get('datasets', {}) or {}).get('seg_general', 'data/segmentation/data.yaml')))
if seg_yaml.exists():
    run_stage('seg_general')
else:
    print('Dataset de segmentação ainda não disponível:', seg_yaml)


## 13. Resultados e pacote pronto para TCC_Analise


In [ ]:
import pandas as pd
summary = Path(str((runtime.get('paths', {}) or {}).get('training_project', 'workspace/training_runs'))) / 'experiment_summary.csv'
if summary.exists(): display(pd.read_csv(summary))
export_dir = Path(str((runtime.get('paths', {}) or {}).get('analysis_export', 'exports/TCC_Analise_models')))
print('Pasta para análise:', export_dir)
print('ZIP:', export_dir.with_suffix('.zip'))
